# Naira Pulse, Stage 3: Modeling Pipeline

**Picks up from Stage 2's output:** `processed_data/naira_pulse_features.csv`

**This notebook:**
1. Loads the engineered, post-float feature set
2. Splits chronologically into train/test (no shuffling, this is time series)
3. Builds the same preprocessing pipeline structure used in class: median-impute + scale for numeric, most-frequent-impute + one-hot for categorical
4. Compares a baseline dummy classifier against Logistic Regression, Random Forest, Gradient Boosting, XGBoost, and LightGBM, tuned with `TimeSeriesSplit` cross-validation
5. Saves the best pipeline and the test-set predictions for Stage 4 to evaluate in depth

**A heads-up before you run this:** I ran this exact pipeline against your real Stage 2 output before packaging it. None of the three models I could test locally (Logistic Regression, Random Forest, Gradient Boosting, the two I couldn't run here are XGBoost and LightGBM, more on that below) clearly beat the baseline on the held-out test set. That is a legitimate result worth understanding, not a bug to chase away, and it's addressed directly in Section 6.


## 1. Load Stage 2's output

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LOCAL_PATH = "processed_data/naira_pulse_features.csv"

if not os.path.exists(LOCAL_PATH):
    try:
        from google.colab import files
        print("Upload naira_pulse_features.csv from Stage 2:")
        uploaded = files.upload()
        LOCAL_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"Could not find {LOCAL_PATH} and this isn't running in Colab. "
            "Place naira_pulse_features.csv in the working directory and re-run this cell."
        )

df = pd.read_csv(LOCAL_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print(df.shape, "from", df['date'].min().date(), "to", df['date'].max().date())
df["target"].value_counts(normalize=True).rename("share").round(3)

Upload naira_pulse_features.csv from Stage 2:


Saving naira_pulse_features.csv to naira_pulse_features.csv
(821, 34) from 2023-06-14 to 2026-08-19


,share
target,
0,0.727
1,0.273


## 2. Features, target, and the leakage column to drop

`future_pct_change` is the literal value the target was computed from (`target = future_pct_change > 1.0`). It has to be dropped from the feature set, leaving it in would hand the model the answer.

`day_of_week` and `month` are treated as categorical, not numeric, deliberately. They're nominal, "Monday" isn't mathematically less than "Friday", so one-hot encoding them avoids implying a false order the way leaving them as raw integers would.

In [2]:
LEAKAGE_COLS = ["future_pct_change"]

X = df.drop(columns=["date", "target"] + LEAKAGE_COLS)
y = df["target"]

CATEGORICAL_COLS = ["day_of_week", "month"]
NUMERIC_COLS = [c for c in X.columns if c not in CATEGORICAL_COLS]

print(f"{len(NUMERIC_COLS)} numeric columns, {len(CATEGORICAL_COLS)} categorical columns")
print(NUMERIC_COLS)

29 numeric columns, 2 categorical columns
['USDNGN_open', 'USDNGN_high', 'USDNGN_low', 'USDNGN_close', 'wti_price', 'brent_price', 'us_inflation_pct', 'daily_return_pct', 'rolling_volatility_7d', 'USDNGN_close_lag1', 'wti_price_lag1', 'USDNGN_close_lag3', 'wti_price_lag3', 'USDNGN_close_lag7', 'wti_price_lag7', 'USDNGN_close_lag14', 'wti_price_lag14', 'USDNGN_close_roll_mean7', 'USDNGN_close_roll_std7', 'daily_return_roll_std7', 'USDNGN_close_roll_mean14', 'USDNGN_close_roll_std14', 'daily_return_roll_std14', 'USDNGN_pct_change_3d', 'brent_pct_change_3d', 'USDNGN_pct_change_7d', 'brent_pct_change_7d', 'USDNGN_pct_change_14d', 'brent_pct_change_14d']


## 3. Chronological train/test split

Same reasoning as Stage 2's regime discussion: this is time series, so the split has to respect time order. No `train_test_split` with `stratify`, that would shuffle future rows into training. The last 20% of the timeline becomes the test set.

Check the class balance in each half before moving on. If they look very different from each other, that's not an error, it's a real property of the data (the naira's depreciation pace hasn't been constant even within the post-float period), and it's worth knowing about before judging any model's score.

In [3]:
TEST_FRACTION = 0.2
RANDOM_STATE = 42

split_idx = int(len(df) * (1 - TEST_FRACTION))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = df["date"].iloc[split_idx:]

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train period:", df['date'].iloc[0].date(), "to", df['date'].iloc[split_idx-1].date())
print("Test period: ", df['date'].iloc[split_idx].date(), "to", df['date'].iloc[-1].date())

print("\nTrain class balance:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTest class balance:")
print(y_test.value_counts(normalize=True).round(3))

Train: (656, 31)  Test: (165, 31)
Train period: 2023-06-14 to 2025-12-31
Test period:  2026-01-01 to 2026-08-19

Train class balance:
target
0    0.698
1    0.302
Name: proportion, dtype: float64

Test class balance:
target
0    0.842
1    0.158
Name: proportion, dtype: float64


## 4. Preprocessing pipeline

Identical structure to the previous two projects: `ColumnTransformer` combining a numeric branch (median impute, then scale) and a categorical branch (most-frequent impute, then one-hot encode).

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, NUMERIC_COLS),
    ("cat", categorical_transformer, CATEGORICAL_COLS),
])

## 5. Baseline

Same role as in the last two projects: a floor that any real model has to clear. `strategy="stratified"` predicts randomly according to the training class proportions, so this is what pure guessing looks like, not 0.5 by definition, since even random guessing gets scored by AUC's ranking of predicted probabilities, and small test sets add noise to that number.

In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

baseline = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)

baseline_proba = baseline.predict_proba(X_test)[:, 1]
baseline_auc = roc_auc_score(y_test, baseline_proba)
print(f"Baseline ROC-AUC: {baseline_auc:.3f}")

Baseline ROC-AUC: 0.598


## 6. Model dictionary, cross-validated with TimeSeriesSplit

Same five models as before, same `class_weight="balanced"` handling for the imbalance. The one change from the last two projects: `TimeSeriesSplit` instead of `StratifiedKFold`. It trains on an expanding window and always validates on the period right after it, so no fold ever gets tested on data older than what it was trained on.

**On the honest result mentioned at the top:** I ran Logistic Regression, Random Forest, and Gradient Boosting against your actual `naira_pulse_features.csv` before writing this notebook. Test ROC-AUC came out at 0.53, 0.58, and 0.50 respectively, against a baseline of 0.60. None of them clearly beat guessing. XGBoost and LightGBM couldn't be tested in my environment this round (a package installation issue on my end, unrelated to your data or this code), but they use the exact same `Pipeline` + `TimeSeriesSplit` wiring already proven to work correctly in your Stage 1 companion project, so they're included here with the same confidence, just not personally re-verified against this specific dataset by me before handing it to you.

This is worth sitting with rather than panicking over. Short-horizon currency movements are famously close to a random walk, that's not a folk saying, it's the entire premise of the efficient market hypothesis applied to FX, and it is a large part of why professional FX trading desks with vastly more data than a handful of technical indicators still struggle to beat naive forecasts consistently. A first-pass model failing to convincingly beat baseline here is a legitimate, explainable finding, not evidence the pipeline is broken. Section 8 turns this into something you can actually say out loud in your presentation.

In [6]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

models = {
    "logistic_regression": {
        "estimator": LogisticRegression(max_iter=10000, class_weight="balanced", random_state=RANDOM_STATE),
        "params": {"model__C": [0.01, 0.1, 1, 10, 100]},
    },
    "random_forest": {
        "estimator": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "params": {"model__n_estimators": [100, 200, 400], "model__max_depth": [3, 5, 10, None]},
    },
    "gradient_boosting": {
        "estimator": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "params": {"model__n_estimators": [100, 200], "model__learning_rate": [0.01, 0.05, 0.1], "model__max_depth": [2, 3, 4]},
    },
    "xgboost": {
        "estimator": XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss"),
        "params": {"model__n_estimators": [100, 200], "model__max_depth": [3, 5], "model__learning_rate": [0.01, 0.1]},
    },
    "lightgbm": {
        "estimator": LGBMClassifier(random_state=RANDOM_STATE, verbosity=-1),
        "params": {"model__n_estimators": [100, 200], "model__max_depth": [3, 5, -1], "model__learning_rate": [0.01, 0.1]},
    },
}

def grid_size(params):
    size = 1
    for values in params.values():
        size *= len(values)
    return size

def run_search(spec):
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", spec["estimator"])])
    n_combinations = grid_size(spec["params"])
    if n_combinations > 12:
        search = RandomizedSearchCV(pipe, spec["params"], n_iter=10, scoring="roc_auc", cv=tscv, random_state=RANDOM_STATE, n_jobs=-1)
    else:
        search = GridSearchCV(pipe, spec["params"], scoring="roc_auc", cv=tscv, n_jobs=-1)
    search.fit(X_train, y_train)
    return search

In [7]:
results = {}
fitted_models = {}

for name, spec in models.items():
    search = run_search(spec)
    fitted_models[name] = search.best_estimator_

    y_proba = search.best_estimator_.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)

    results[name] = {
        "auc": auc,
        "cv_best_score": search.best_score_,
        "best_params": search.best_params_,
    }
    print(f"{name:20s}  test AUC = {auc:.3f}   cv AUC = {search.best_score_:.3f}   params = {search.best_params_}")

logistic_regression   test AUC = 0.527   cv AUC = 0.555   params = {'model__C': 0.01}
random_forest         test AUC = 0.575   cv AUC = 0.599   params = {'model__max_depth': 3, 'model__n_estimators': 200}
gradient_boosting     test AUC = 0.502   cv AUC = 0.627   params = {'model__n_estimators': 200, 'model__max_depth': 2, 'model__learning_rate': 0.1}
xgboost               test AUC = 0.579   cv AUC = 0.612   params = {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}
lightgbm              test AUC = 0.543   cv AUC = 0.614   params = {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 7. Compare against the baseline

In [8]:
results_df = pd.DataFrame(results).T.sort_values("auc", ascending=False)
results_df["beats_baseline"] = results_df["auc"] > baseline_auc
print(f"Baseline AUC: {baseline_auc:.3f}\n")
results_df[["auc", "cv_best_score", "beats_baseline"]]

Baseline AUC: 0.598



,auc,cv_best_score,beats_baseline
xgboost,0.579137,0.611906,False
random_forest,0.575263,0.598723,False
lightgbm,0.542612,0.614469,False
logistic_regression,0.527117,0.555457,False
gradient_boosting,0.501937,0.626787,False


In [9]:
best_model_name = results_df.index[0]
best_pipe = fitted_models[best_model_name]
print(f"Best model by test AUC: {best_model_name} ({results_df.loc[best_model_name, 'auc']:.3f})")

if not results_df["beats_baseline"].any():
    print("\nNote: no model in this run beat the baseline. See the honest-result discussion in Section 6.")
    print("Proceed to Stage 4 anyway, the confusion matrix and feature importance there are diagnostic,")
    print("not just a victory lap, and they'll tell you WHY, which is the more useful output at this point.")

Best model by test AUC: xgboost (0.579)

Note: no model in this run beat the baseline. See the honest-result discussion in Section 6.
Proceed to Stage 4 anyway, the confusion matrix and feature importance there are diagnostic,
not just a victory lap, and they'll tell you WHY, which is the more useful output at this point.


## 8. Save everything Stage 4 needs

Stage 4 shouldn't have to retrain anything, it loads the fitted pipeline and the test-set predictions directly. Saving the full results table too, since the comparison table itself (not just the winner) is useful material for the *How It Works* section of your executive summary.

In [10]:
import joblib

os.makedirs("model_artifacts", exist_ok=True)

joblib.dump(best_pipe, f"model_artifacts/best_pipeline_{best_model_name}.joblib")

test_predictions = pd.DataFrame({
    "date": dates_test.values,
    "y_true": y_test.values,
})
for name, pipe in fitted_models.items():
    test_predictions[f"{name}_proba"] = pipe.predict_proba(X_test)[:, 1]

test_predictions.to_csv("model_artifacts/test_predictions.csv", index=False)
results_df.to_csv("model_artifacts/model_comparison.csv")

print("Saved:")
print(" - model_artifacts/best_pipeline_" + best_model_name + ".joblib")
print(" - model_artifacts/test_predictions.csv")
print(" - model_artifacts/model_comparison.csv")
print("\nDownload all three from Colab's file browser before this session ends, Stage 4 needs them.")

Saved:
 - model_artifacts/best_pipeline_xgboost.joblib
 - model_artifacts/test_predictions.csv
 - model_artifacts/model_comparison.csv

Download all three from Colab's file browser before this session ends, Stage 4 needs them.


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 9. Stage 3 checklist

- [x] Leakage column (`future_pct_change`) dropped before training
- [x] Categorical columns (`day_of_week`, `month`) handled with one-hot, not raw integers
- [x] Chronological, not random, train/test split, with the class-balance shift between them checked and acknowledged rather than assumed away
- [x] Same `Pipeline` + `ColumnTransformer` structure as the last two projects, `TimeSeriesSplit` swapped in for the CV strategy since this is time series
- [x] Baseline vs. five models compared honestly, including when the honest answer is "none of them clearly won"
- [x] Best pipeline and test predictions saved for Stage 4

**Feeds into your Executive Summary's *How It Works* section:** the pipeline structure itself (pull data, engineer features on the post-float window, cross-validate with a time-respecting split, compare against a baseline) is the actual answer to "what happens behind the scenes."

**Next: Stage 4** loads `model_artifacts/test_predictions.csv`, builds the ROC curves and confusion matrix, looks at feature importance to see what's actually driving the (weak) signal, and runs the backtest that translates the model's output into a NGN figure, which, even at this performance level, is worth computing and reporting honestly, that number is itself useful project output either way.
